In [0]:
from pyspark.sql import functions as F

df_tempo = (

    spark.range(1)

    .select(

        F.explode(

            F.sequence(

                F.to_date(
                    F.lit("2010-01-01")
                ),

                F.to_date(
                    F.lit("2035-12-31")
                ),

                F.expr("interval 1 day")

            )

        ).alias("data")

    )

)

In [0]:
df_tempo = (

    df_tempo

    .withColumn(
        "sk_data",
        F.date_format(
            "data",
            "yyyyMMdd"
        ).cast("int")
    )

    .withColumn(
        "ano",
        F.year("data")
    )

    .withColumn(
        "mes",
        F.month("data")
    )

    .withColumn(
        "nome_mes",
        F.date_format(
            "data",
            "MMMM"
        )
    )

    .withColumn(
        "trimestre",
        F.quarter("data")
    )

    .withColumn(
        "semana_ano",
        F.weekofyear("data")
    )

    .withColumn(
        "dia_mes",
        F.dayofmonth("data")
    )

    .withColumn(
        "dia_semana",
        F.dayofweek("data")
    )

    .withColumn(
        "nome_dia_semana",
        F.date_format(
            "data",
            "EEEE"
        )
    )

    .withColumn(

        "flag_final_semana",

        F.when(
            F.dayofweek("data").isin([1,7]),
            1
        ).otherwise(0)

    )

    .withColumn(
        "dt_criacao_gold",
        F.current_timestamp()
    )

)

In [0]:
display(df_tempo)

In [0]:
(
    df_tempo
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.dim_tempo"
    )
)

In [0]:
%sql
OPTIMIZE workspace.gold.dim_tempo
ZORDER BY (
    data,
    ano,
    mes
)

In [0]:
%sql
SELECT *
FROM workspace.gold.dim_tempo
LIMIT 20